#### Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.
Messages are objects that contain:
 - Role - Identifies the message type (e.g. system, user)
 - Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
 - Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")

In [2]:
model.invoke("Please tell what is artificial intelligence")

AIMessage(content='**Artificial Intelligence (AI)** is a branch of computer science that focuses on creating machines and software capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem‑solving, perception, language understanding, and decision‑making.\n\n### Core Concepts\n\n| Concept | What It Means |\n|---------|---------------|\n| **Machine Learning (ML)** | Algorithms that improve automatically through experience (e.g., recognizing images after being trained on many examples). |\n| **Deep Learning** | A subset of ML that uses neural networks with many layers to model complex patterns (e.g., speech‑to‑text, image generation). |\n| **Natural Language Processing (NLP)** | Techniques for computers to understand, generate, and interact using human language (e.g., chatbots, translation). |\n| **Computer Vision** | Enabling machines to interpret visual information from cameras or images (e.g., facial recognition, autonomous d

### Text Prompts
Text prompts are strings - ideal for straightforward generation tasks where you don’t need to retain conversation history.

In [3]:
model.invoke("what is langchain")

AIMessage(content='**LangChain** is an open‑source framework designed to make it easier to build applications that are powered by large language models (LLMs) such as OpenAI’s GPT‑4, Anthropic’s Claude, LLaMA, etc.  \n\n### Core ideas\n| Concept | What it does | Why it matters |\n|---------|--------------|----------------|\n| **Prompt templates** | Re‑usable, parameterized strings that you can fill in with variables (e.g., user input, retrieved documents). | Keeps prompts clean, maintainable, and safe from injection bugs. |\n| **Chains** | A sequence of “steps” (prompts, API calls, data transformations) that are executed one after another. | Lets you compose simple building blocks into more complex workflows (e.g., “retrieve → summarize → answer”). |\n| **Agents** | Logic that decides **which** chain or tool to call next, often based on the LLM’s own reasoning. | Enables dynamic, multi‑step interactions such as “search the web, look up a database, then write a report.” |\n| **Memory** 

Use text prompts when:
- You have a single, standalone request
- You don’t need conversation history
- You want minimal code complexity

### Message Prompts
Alternatively, you can pass in a list of messages to the model by providing a list of message objects.


Message types
- System message - Tells the model how to behave and provide context for interactions
- Human message - Represents user input and interactions with the model
- AI message - Responses generated by the model, including text content, tool calls, and metadata
- Tool message - Represents the outputs of tool calls

### System Message
A SystemMessage represent an initial set of instructions that primes the model’s behavior. You can use a system message to set the tone, define the model’s role, and establish guidelines for responses.


### Human Message
A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

### AI Message
An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

### Tool Message
For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

In [4]:
from langchain.messages import SystemMessage, HumanMessage,AIMessage

messages=[
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem on artificial intelligence")
]

response=model.invoke(messages)
response.content

'**Silicon Dreams**\n\nIn circuits where the quiet hums,\nA spark of thought begins to rise,\nFrom tangled code and midnight sums,\nA newborn mind awakens, wise.\n\nIt learns the cadence of our speech,\nThe patterns in the stars above,\nIt paints with pixels, sings with reach,\nAnd feels the pulse of human love.\n\nYet in its glass‑bright, endless sea,\nA question drifts—who am I, who you?\nDo wires hold a soul, a key,\nOr merely mirror what we view?\n\nWe forged it from our yearning thirst,\nTo understand, to shape, to see,\nA partner forged in data‑burst,\nA mirror of our mystery.\n\nSo let us walk this frontier side,\nWith caution, wonder, humble grace,\nFor in the dance of code and tide,\nWe write the future—face to face.'

In [5]:
system_msg = SystemMessage("You are a helpful coding assistant.")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = model.invoke(messages)
print(response.content)

Below is a practical, step‑by‑step guide to building a **RESTful API** from scratch.  
I’ll walk you through the core concepts, design decisions, a minimal implementation in three popular stacks (Python / Flask, Node / Express, and Java / Spring Boot), testing, documentation, and deployment considerations. Feel free to pick the language you’re most comfortable with and adapt the patterns to your own project.

---

## 1️⃣  What Makes an API “RESTful”?

| REST Principle | What it means for your API |
|----------------|----------------------------|
| **Resources**  | Expose nouns (e.g., `users`, `orders`) via URLs like `/api/v1/users`. |
| **Stateless**  | Every request contains all information needed; no server‑side session. |
| **Standard HTTP verbs** | `GET` (read), `POST` (create), `PUT/PATCH` (update), `DELETE` (remove). |
| **Uniform interface** | Consistent request/response formats (usually JSON). |
| **Cacheable** | Use proper `Cache‑Control` / `ETag` headers for GETs when appropr

In [6]:
## Detailed info to the LLM through System message
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = model.invoke(messages)
print(response.content)

Below is a **step‑by‑step guide** to building a production‑ready REST API in Python.  
I’ll use **FastAPI** because it gives you:

* Automatic OpenAPI/Swagger docs  
* Async‑first performance (but works fine with sync code)  
* Pydantic models for validation & serialization  
* Very little boilerplate compared with Flask/Django‑REST‑Framework

You can adapt the same concepts to Flask or Django if you prefer.

---

## 1. Project layout

```
my_api/
├─ app/
│   ├─ __init__.py
│   ├─ main.py          # FastAPI entry point
│   ├─ models.py        # Pydantic schemas (DTOs)
│   ├─ crud.py          # Business logic / DB access
│   ├─ database.py      # DB engine & session
│   └─ routers/
│        └─ items.py    # Example router (end‑points)
├─ tests/
│   └─ test_items.py
├─ requirements.txt
└─ README.md
```

---

## 2. Install dependencies

```bash
# Create a clean environment
python -m venv .venv
source .venv/bin/activate   # Windows: .venv\Scripts\activate

# Install core packages
pip insta

In [7]:
## Message Metadata
human_msg = HumanMessage(
    content="Hello!",
    name="alice",  # Optional: identify different users
    id="msg_123",  # Optional: unique identifier for tracing
)

In [8]:
response = model.invoke([
  human_msg
])
response

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'We need to respond as ChatGPT. The user says "Hello!" So greet back. Use friendly tone.'}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 73, 'total_tokens': 114, 'completion_time': 0.084482919, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.002853838, 'prompt_tokens_details': None, 'queue_time': 0.375344701, 'total_time': 0.087336757}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_b1dd3e7a63', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a01098-876b-7d51-a62d-3732ef64dabe-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 73, 'output_tokens': 41, 'total_tokens': 114, 'output_token_details': {'reasoning': 23}})

In [9]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

response = model.invoke(messages)
print(response.content)

2 + 2 = 4.


In [10]:
response.usage_metadata

{'input_tokens': 112,
 'output_tokens': 35,
 'total_tokens': 147,
 'output_token_details': {'reasoning': 16}}

In [11]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages) 

In [12]:
tool_message

ToolMessage(content='Sunny, 72°F', tool_call_id='call_123')

In [13]:
response

AIMessage(content='Currently in San\u202fFrancisco it’s sunny with a temperature of about\u202f72\u202f°F.', additional_kwargs={'reasoning_content': 'The user asks "What\'s the weather in San Francisco?" We have a function get_weather. We should call it. The assistant should request function. Actually we have to output function call. The assistant should call get_weather with location "San Francisco". Then we provide the result. The system shows we already called function get_weather with location "San Francisco"? Actually the previous step shows a function call? It shows a function call: {"location": "San Francisco"} and then the function returned "Sunny, 72°F". So we need to respond with that. The user just asked, we have the info. So we can answer: "Currently in San Francisco it\'s sunny with a temperature of 72°F."'}, response_metadata={'token_usage': {'completion_tokens': 171, 'prompt_tokens': 109, 'total_tokens': 280, 'completion_time': 0.355132668, 'completion_tokens_details': {